# Neuro-Symbolic Clinical Dysbiosis Risk Assessment Pipeline
### System Overview
This notebook implements a multi-agent clinical decision support system. It combines deep learning time-series forecasting (BiLSTM, GRU-Attention, TFT) with a LLM-based 'Arbiter' using dynamic calibration math (ECE) to interpret microbiome trajectories.

In [1]:
# =====================================================
# GLOBAL DEPENDENCIES & ENVIRONMENT SETUP
# =====================================================
try:
    import tensorflow as tf
except ImportError:
    print("TensorFlow not found. Installing...")
    !pip install -q tensorflow
    import tensorflow as tf

import os
import json
import zipfile
import warnings
import numpy as np
import pandas as pd
import torch
import gc

# Machine Learning & Signal Processing
from tensorflow.keras.layers import Input, Dense, Layer, LayerNormalization, Dropout, Add
from tensorflow.keras.models import Model, load_model
import tensorflow.keras.backend as K
import keras

# Evaluation & Preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.calibration import calibration_curve
from sklearn.metrics import classification_report, matthews_corrcoef, roc_auc_score

# =====================================================
# MLOps GLOBAL CONFIGURATIONS & HARDWARE MANAGEMENT
# =====================================================
SEQ_LEN = 14
RANDOM_STATE = 42
ECE_UNCERTAINTY_THRESHOLD = 0.05

# -----------------------------------------------------
# CRITICAL OPTIMIZATION: Prevent TF from hoarding VRAM
# -----------------------------------------------------
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Hardware Optimization: Dynamic VRAM growth enabled for {len(gpus)} GPU(s).")
    except RuntimeError as e:
        print(f"Hardware Optimization Error: {e}")

# Suppress non-critical warnings
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = 'true'
os.environ['HF_HUB_DISABLE_AUTO_CONVERSION'] = '1'

print(f"Environment initialized: TensorFlow {tf.__version__} and MLOps configurations synchronized.")

Hardware Optimization: Dynamic VRAM growth enabled for 1 GPU(s).
Environment initialized: TensorFlow 2.20.0 and MLOps configurations synchronized.


### 1. File Extraction and Data Loading
Extracts the raw microbiome dataset and initializes the primary DataFrame.

In [2]:
# File I/O operations (Imports moved to global header)
zip_file_path = '/content/asv_interpretability_dataset_modified.zip'
extract_dir = '/content/'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

csv_file_path = os.path.join(extract_dir, 'asv_interpretability_dataset_modified.csv')
df_benchmark = pd.read_csv(csv_file_path, dtype={'PatientID': str})

### 2. Clinical Data Normalization
Standardizes complex clinical strings (e.g., neutrophil counts) into numeric types for ML compatibility.

In [3]:
# --- Handle NeutrophilCount with '<0.1' values ---
def parse_neutrophil(value):
    try:
        return float(value)
    except:
        if isinstance(value, str) and "<" in value:
            threshold = float(value.replace("<", "").strip())
            return threshold / 2  # or use threshold itself
        return np.nan

df_benchmark['NeutrophilCount'] = df_benchmark['NeutrophilCount'].apply(parse_neutrophil)

### 3. Feature Engineering: Stool Consistency
Applies one-hot encoding to qualitative stool consistency observations.

In [4]:
# One-hot encode stool consistency
df_benchmark = pd.get_dummies(df_benchmark, columns=['Consistency'])

### 4. Patient-Level Dysbiosis Labeling
Implements the clinical logic required to label dysbiosis based on temporal persistence across patient trajectories.

In [5]:
# Step 1: Apply your original function
def label_dysbiosis(row):
    is_temp_abnormal = row['MaxTemperature'] > 38.0
    is_neutro_low = row['NeutrophilCount'] < 500
    is_consistency_liquid = row.get('Consistency_liquid', 0) == 1
    return int(is_temp_abnormal and is_neutro_low and is_consistency_liquid)

df_benchmark['RowLabel'] = df_benchmark.apply(label_dysbiosis, axis=1)

# Step 2: Sort data by patient and time
df_benchmark = df_benchmark.sort_values(["PatientID", "DayRelativeToNearestHCT"])

# Step 3: Define patient-level trajectory labeling
def patient_dysbiosis(row_labels, min_consecutive=2):
    consec = 0
    for val in row_labels:
        if val == 1:
            consec += 1
            if consec >= min_consecutive:
                return 1  # patient had dysbiosis
        else:
            consec = 0
    return 0  # no persistent dysbiosis

# Step 4: Apply per patient
patient_labels = df_benchmark.groupby("PatientID")['RowLabel'].apply(patient_dysbiosis).reset_index(name="DysbiosisLabel")
df_benchmark = df_benchmark.merge(patient_labels, on="PatientID", how="left")

### 5. Microbiome Centered Log-Ratio (CLR) Transformation
Applies CLR to handle the compositional nature of relative abundance data.

In [6]:
# Keep metadata along with abundances
metadata_cols = ['PatientID', 'SampleID', 'DayRelativeToNearestHCT',
                 'MaxTemperature', 'NeutrophilCount'] + \n                [col for col in df_benchmark.columns if col.startswith('Consistency_')] + \n                ['DysbiosisLabel']

# Pivot genus abundances (before CLR)
genus_pivot = df_benchmark.pivot_table(
    index=['PatientID', 'SampleID', 'DayRelativeToNearestHCT'],
    columns='Genus',
    values='RelativeAbundance',
    fill_value=0
).reset_index()

# Merge with metadata (before CLR)
metadata = df_benchmark[metadata_cols].drop_duplicates(subset=['PatientID', 'SampleID', 'DayRelativeToNearestHCT'])
merged_df_benchmark = pd.merge(genus_pivot, metadata, on=['PatientID', 'SampleID', 'DayRelativeToNearestHCT'], how='left')

# ==================== Apply Centered Log-Ratio (CLR) Transformation ====================
def clr_transform(data, pseudocount=1):
    """Applies Centered Log-Ratio (CLR) transformation with a pseudocount."""
    data_np = data.to_numpy()
    data_with_pseudocount = data_np + pseudocount
    log_data = np.log(data_with_pseudocount)
    geometric_mean = np.mean(log_data, axis=1, keepdims=True)
    clr_data = log_data - geometric_mean
    return pd.DataFrame(clr_data, columns=data.columns, index=data.index)

original_genus_cols = df_benchmark['Genus'].unique().tolist()
genus_cols_in_merged = [col for col in original_genus_cols if col in merged_df_benchmark.columns]

merged_df_benchmark[genus_cols_in_merged] = clr_transform(merged_df_benchmark[genus_cols_in_merged])
print("Applied CLR transformation to genus abundance columns.")

Applied CLR transformation to genus abundance columns.


### 6. Temporal Sequence Generation and Stratified Splitting
Partitions data into Train/Val/Test sets at the patient level and builds sliding-window sequences for time-series models.

In [7]:
# # Sort & reshape by patient and day
# merged_df_benchmark = merged_df_benchmark.sort_values(['PatientID', 'DayRelativeToNearestHCT'])

# def build_sequences_with_labels(df, feature_cols, seq_len=14):
#     X, y, index_info = [], [], []
#     for pid, group in df.groupby('PatientID'):
#         group = group.sort_values('DayRelativeToNearestHCT')
#         data = group[feature_cols].values
#         labels = group['DysbiosisLabel'].values

#         for i in range(len(group) - seq_len + 1):
#             X.append(data[i:i+seq_len])
#             y.append(labels[i+seq_len-1])
#             index_info.append((pid, group.iloc[i+seq_len-1]['DayRelativeToNearestHCT']))

#     return np.array(X), np.array(y), index_info

# # Patient-Level Stratified Split
# patient_ids = merged_df_benchmark['PatientID'].unique()
# patient_dysbiosis_status = merged_df_benchmark.groupby('PatientID')['DysbiosisLabel'].max() > 0
# patient_stratification_key = patient_dysbiosis_status.loc[patient_ids].values

# sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
# train_patient_idx, temp_patient_idx = next(sss1.split(patient_ids, patient_stratification_key))

# train_patient_ids = patient_ids[train_patient_idx]
# temp_patient_ids = patient_ids[temp_patient_idx]
# temp_stratification_key = patient_stratification_key[temp_patient_idx]

# sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
# val_patient_idx, test_patient_idx = next(sss2.split(temp_patient_ids, temp_stratification_key))

# val_patient_ids = temp_patient_ids[val_patient_idx]
# test_patient_ids = temp_patient_ids[test_patient_idx]

# print(f"Train patients: {len(train_patient_ids)} | Val patients: {len(val_patient_ids)} | Test patients: {len(test_patient_ids)}")

# # Build Sequences Per Split
# all_potential_feature_cols = [col for col in merged_df_benchmark.columns if col not in ['PatientID','SampleID', 'DayRelativeToNearestHCT', 'DysbiosisLabel']]

# train_df = merged_df_benchmark[merged_df_benchmark['PatientID'].isin(train_patient_ids)].copy()
# X_train_raw_all, y_train, _ = build_sequences_with_labels(train_df, all_potential_feature_cols, seq_len=14)

# val_df = merged_df_benchmark[merged_df_benchmark['PatientID'].isin(val_patient_ids)].copy()
# X_val_raw_all, y_val, _ = build_sequences_with_labels(val_df, all_potential_feature_cols, seq_len=14)

# test_df = merged_df_benchmark[merged_df_benchmark['PatientID'].isin(test_patient_ids)].copy()
# X_test_raw_all, y_test, _ = build_sequences_with_labels(test_df, all_potential_feature_cols, seq_len=14)

# # Variance Thresholding (Train Only)
# original_genus_cols = df_benchmark['Genus'].unique().tolist()
# genus_col_indices_in_all = [all_potential_feature_cols.index(col) for col in original_genus_cols if col in all_potential_feature_cols]

# X_train_flat_all = X_train_raw_all.reshape(-1, X_train_raw_all.shape[-1])
# X_train_genus_flat = X_train_flat_all[:, genus_col_indices_in_all]

# train_genus_variances = np.var(X_train_genus_flat, axis=0)
# non_zero_var_genus_indices_in_genus_subset = np.where(train_genus_variances > 1e-6)[0]
# genus_cols_to_keep = [original_genus_cols[i] for i in non_zero_var_genus_indices_in_genus_subset]

# other_feature_cols = [col for col in all_potential_feature_cols if col not in original_genus_cols]
# final_feature_cols = genus_cols_to_keep + other_feature_cols
# final_feature_indices_in_all = [all_potential_feature_cols.index(col) for col in final_feature_cols]

# X_train_raw = X_train_raw_all[:, :, final_feature_indices_in_all]
# X_val_raw   = X_val_raw_all[:, :, final_feature_indices_in_all]
# X_test_raw  = X_test_raw_all[:, :, final_feature_indices_in_all]

# # Scaling (Train Fitted)
# scaler = MinMaxScaler()
# X_train_scaled = scaler.fit_transform(X_train_raw.reshape(-1, X_train_raw.shape[-1])).reshape(X_train_raw.shape)
# X_val_scaled   = scaler.transform(X_val_raw.reshape(-1, X_val_raw.shape[-1])).reshape(X_val_raw.shape)
# X_test_scaled  = scaler.transform(X_test_raw.reshape(-1, X_test_raw.shape[-1])).reshape(X_test_raw.shape)

# X_train, X_val, X_test = X_train_scaled, X_val_scaled, X_test_scaled
# n_features = X_train.shape[2]

In [ ]:
import gc
import numpy as np

# ==============================================================
# OPTIMIZATION 1: Downcast to Float32 to instantly halve RAM usage
# ==============================================================
# Identify potential feature columns
all_potential_feature_cols = [col for col in merged_df_benchmark.columns
                              if col not in ['PatientID','SampleID', 'DayRelativeToNearestHCT', 'DysbiosisLabel']]

# Sort by patient and day
merged_df_benchmark = merged_df_benchmark.sort_values(['PatientID', 'DayRelativeToNearestHCT'])

# ==============================================================
# Patient-Level Stratified Split
# ==============================================================
patient_ids = merged_df_benchmark['PatientID'].unique()
patient_dysbiosis_status = merged_df_benchmark.groupby('PatientID')['DysbiosisLabel'].max() > 0
patient_stratification_key = patient_dysbiosis_status.loc[patient_ids].values

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_patient_idx, temp_patient_idx = next(sss1.split(patient_ids, patient_stratification_key))

train_patient_ids = patient_ids[train_patient_idx]
temp_patient_ids = patient_ids[temp_patient_idx]
temp_stratification_key = patient_stratification_key[temp_patient_idx]

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
val_patient_idx, test_patient_idx = next(sss2.split(temp_patient_ids, temp_stratification_key))

val_patient_ids = temp_patient_ids[val_patient_idx]
test_patient_ids = temp_patient_ids[test_patient_idx]

print(f"Train patients: {len(train_patient_ids)} | Val patients: {len(val_patient_ids)} | Test patients: {len(test_patient_ids)}")

# ==============================================================
# OPTIMIZATION 2: Align Features with Pre-Trained Models
# ==============================================================
original_genus_cols = df_benchmark['Genus'].unique().tolist()
genus_cols_in_all = [col for col in original_genus_cols if col in all_potential_feature_cols]

# Isolate just the training rows (2D) to calculate variance
train_mask = merged_df_benchmark['PatientID'].isin(train_patient_ids)
# Use raw numpy array to match original variance calculation (ddof=0)
train_genus_data = merged_df_benchmark.loc[train_mask, genus_cols_in_all].values

# Calculate population variance
train_genus_variances = np.var(train_genus_data, axis=0)

# MLOps Strict Enforcement: The pre-trained models expect exactly 141 features.
other_feature_cols = [col for col in all_potential_feature_cols if col not in original_genus_cols]
TARGET_TOTAL_FEATURES = 141
target_genus_count = TARGET_TOTAL_FEATURES - len(other_feature_cols)

# Sort variances and take the exact top 'target_genus_count' to ensure shape match
# We use argsort, grab the top N, and sort the indices to maintain original column order
top_genus_indices = np.argsort(train_genus_variances)[-target_genus_count:]
top_genus_indices = np.sort(top_genus_indices)

genus_cols_to_keep = [genus_cols_in_all[i] for i in top_genus_indices]
final_feature_cols = genus_cols_to_keep + other_feature_cols

print(f"Aligned features to model spec: {len(final_feature_cols)} (Genus kept: {len(genus_cols_to_keep)}, Metadata: {len(other_feature_cols)})")

# Free up memory immediately
del train_genus_data
gc.collect()

# ==============================================================
# OPTIMIZATION 3: Memory-Safe Sequence Builder
# Pre-allocates exact numpy array size instead of appending to lists
# ==============================================================